# Clinical-LLM — QLoRA fine-tune on a Colab GPU

End-to-end: install → build a medical instruction corpus → **QLoRA fine-tune** 
`Qwen2.5-3B-Instruct` → evaluate base vs. fine-tuned on medical-QA benchmarks → 
chat with the result.

> ⚠️ **Research/education only — not medical advice, not a medical device.** 
> Do not enter real patient data (PHI).

**Before you run anything:** switch to a GPU runtime — 
**Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is free and works).

## 0. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU'
print('CUDA device:', torch.cuda.get_device_name(0))

## 1. Get the code

Clones the repo. If you pushed under a different owner/name, edit `REPO_URL`.

In [ ]:
REPO_URL = "https://github.com/asongwe-mutah/clinical-llm.git"
import os
if not os.path.exists('clinical-llm'):
    !git clone $REPO_URL
%cd clinical-llm
!git log --oneline -1

## 2. Install dependencies

Installs torch/transformers/trl/peft + **bitsandbytes** (the 4-bit backend that 
makes QLoRA possible — CUDA only, which is why this needs the GPU runtime).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
# quick sanity check that the 4-bit backend is present
import bitsandbytes as bnb; print('bitsandbytes', bnb.__version__)

## 3. Build the instruction corpus

Downloads three **public** medical-QA datasets (PubMedQA, MedMCQA, MedQuAD) and 
normalizes them into one chat-format corpus. The Colab config caps each source at 
8k examples so this finishes in a couple of minutes; edit `configs/data_colab.yaml` 
to lift the cap on a bigger GPU.

In [ ]:
!python -m clinical_llm.data.prepare --config configs/data_colab.yaml
!wc -l data/processed/train.jsonl data/processed/val.jsonl

In [ ]:
# peek at one formatted example
import json
with open('data/processed/train.jsonl') as f:
    rec = json.loads(f.readline())
for m in rec['messages']:
    print(f"--- {m['role'].upper()} ---\n{m['content'][:300]}\n")

## 4. QLoRA fine-tune

Trains for a bounded `max_steps: 500` (~45–90 min on a T4) so you get a result in 
one session. For a stronger model, open `configs/train_colab.yaml` and set 
`max_steps: -1` (train full epochs) and/or raise `max_seq_len` on an L4/A100.

In [ ]:
!python -m clinical_llm.train.train_qlora --config configs/train_colab.yaml

## 5. Evaluate — base vs. fine-tuned

Scores accuracy by letter-constrained log-likelihood on held-out MedMCQA + PubMedQA. 
`--max-items 300` keeps eval quick; raise for tighter estimates. The printed 
`delta` is your headline number for the model card.

In [ ]:
!python -m clinical_llm.eval.run_eval \
    --base Qwen/Qwen2.5-3B-Instruct \
    --adapter outputs/clinical-qlora \
    --benchmarks medmcqa pubmedqa \
    --max-items 300 \
    --out reports/eval.json

import json; print(json.dumps(json.load(open('reports/eval.json')).get('deltas', {}), indent=2))

## 6. Chat with the fine-tuned model

In [ ]:
from clinical_llm.serve.inference import ClinicalLLM, GenerationSettings
llm = ClinicalLLM(base_model='Qwen/Qwen2.5-3B-Instruct', adapter='outputs/clinical-qlora')

for q in [
    'What is the first-line treatment for anaphylaxis, and by what route?',
    'Explain the difference between type 1 and type 2 respiratory failure.',
]:
    print('Q:', q)
    print('A:', llm.generate(q, settings=GenerationSettings(max_new_tokens=200, temperature=0.3)))
    print('-' * 80)

## 7. Save your adapter

The LoRA adapter is tiny (tens of MB). Download it, or mount Google Drive to keep it.

In [ ]:
# Option A: download a zip of the adapter
!cd outputs && zip -qr clinical-qlora.zip clinical-qlora
from google.colab import files
files.download('outputs/clinical-qlora.zip')

In [ ]:
# Option B: save to Google Drive instead
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r outputs/clinical-qlora /content/drive/MyDrive/

---
Next: paste the eval deltas into `docs/MODEL_CARD.md`, and serve the adapter locally 
with `CLINICAL_LLM_ADAPTER=outputs/clinical-qlora uvicorn clinical_llm.serve.app:app`.